In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_data, build_index

In [2]:
load_dotenv()
openai_client = OpenAI()

In [3]:
data = load_data()
index = build_index(data)

In [ ]:
instructions = """
You are learning assistant.
Your task is to answer questions based on the provided context.
Use the context to find relevant information and provide accurate answers.
Use only the facts from the CONTEXT when answering the QUESTION.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
assistant.rag('What is regularization')

'Regularization is a technique used to avoid overfitting by making the model more simple.'

In [6]:
assistant.rag('What is reularization')

'Regularization is a technique used in machine learning to help prevent overfitting by controlling model complexity. It adds a constraint or penalty so the model does not fit the training data too closely.\n\nThe provided context does not include a specific definition of regularization.'

In [20]:
messages = [
    {"role": "user", "content": "What clustering algorithms are described in the ds_qa database?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

I don’t have direct access to your `ds_qa` database contents from here, so I can’t tell which clustering algorithms are described **unless you share the schema/data or let me inspect it**.

If you want, I can help you find them quickly. Here are a few common ways:

### If you have a table of docs/QA pairs
Run something like:

```sql
SELECT *
FROM your_table
WHERE LOWER(answer) LIKE '%clustering%'
   OR LOWER(question) LIKE '%clustering%'
   OR LOWER(content) LIKE '%clustering%';
```

To specifically identify algorithm names, search for terms like:
- k-means / kmeans
- hierarchical clustering
- DBSCAN
- OPTICS
- Gaussian mixture models
- spectral clustering
- mean shift
- BIRCH
- affinity propagation

### If you want, paste:
- the relevant table names/schema, or
- a sample of the `ds_qa` rows, or
- the output of a search query

and I can extract the clustering algorithms described there.


In [21]:
def search(query):
    boost_dict = {"question": 3.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [22]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the ds_qa database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the ds_qa database."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [23]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
#    tool_choice="required"
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"clustering algorithms described"}', call_id='call_nd3x5Glg2azN9UjugA8JY6ZW', name='search', type='function_call', id='fc_0cba61974d084ba3006a3beb93607881a38fa2a60682dabad7', namespace=None, status='completed')]


In [24]:
import json


call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [25]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [26]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

The ds_qa database describes these clustering algorithms:

- **K-Means**
- **MiniBatch KMeans** (MiniBatchKMeans)
- **BIRCH** (Balanced Iterative Reducing and Clustering using Hierarchies)

It also mentions clustering evaluation metrics, such as **Silhouette Coefficient** and **Dunn’s Index**, but those are not clustering algorithms.


In [27]:
usage = response.usage
input_tokens, output_tokens = usage.input_tokens, usage.output_tokens

In [28]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(input_tokens, output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0002172


The logic behind:
1. Make a call to the LLM <-- we pay
2. LLM decides to call the search tool ('params')
3. We invoke the search, we get the results
4. Send the request back to the LLM (another call) <-- we pay
5. LLM processes the results
6a. LLM gives the answer

6b. LLM decides to make another tool call
7. Send one more API request <-- we pay

In [31]:
response.output

[ResponseOutputMessage(id='msg_0cba61974d084ba3006a3beb9df56081a3942a9cc2428d3d90', content=[ResponseOutputText(annotations=[], text='The ds_qa database describes these clustering algorithms:\n\n- **K-Means**\n- **MiniBatch KMeans** (MiniBatchKMeans)\n- **BIRCH** (Balanced Iterative Reducing and Clustering using Hierarchies)\n\nIt also mentions clustering evaluation metrics, such as **Silhouette Coefficient** and **Dunn’s Index**, but those are not clustering algorithms.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [ ]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

for item in response.output:
   # print(item)

    if item.type == 'function_call':
        # process function call
        print(f'function call: {item.name}, {item.arguments}')
        call_output = make_call(call)
        messages.append(call_output)
        
    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ASSISTANT:
The ds_qa database describes these clustering algorithms:

- **K-Means**
- **MiniBatch KMeans** (MiniBatchKMeans)
- **BIRCH** (Balanced Iterative Reducing and Clustering using Hierarchies)

It also mentions clustering evaluation metrics, such as **Silhouette Coefficient** and **Dunn’s Index**, but those are not clustering algorithms.


In [33]:
messages

[{'role': 'user',
  'content': 'What clustering algorithms are described in the ds_qa database?'},
 ResponseFunctionToolCall(arguments='{"query":"clustering algorithms described"}', call_id='call_nd3x5Glg2azN9UjugA8JY6ZW', name='search', type='function_call', id='fc_0cba61974d084ba3006a3beb93607881a38fa2a60682dabad7', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_nd3x5Glg2azN9UjugA8JY6ZW',
  'output': '[\n  {\n    "id": "47",\n    "category": "ml",\n    "question": "Discuss two clustering algorithms that can scale to large datasets",\n    "answer": "**Minibatch Kmeans:**  Instead of using the full dataset at each iteration, the algorithm\\nis capable of using mini-batches, moving the centroids just slightly at each iteration.\\nThis speeds up the algorithm typically by a factor of 3 or 4 and makes it\\npossible to cluster huge datasets that do not fit in memory. Scikit-Learn implements\\nthis algorithm in the MiniBatchKMeans class.\\n\\n**Bal